### Part 1 - Automated stat analyzer

In [1]:
import pandas as pd
import numpy as np


def automated_stat_analyzer(df, column_name):
    """
    Analyze a DataFrame column and return basic statistical information.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to analyze.

    Returns:
        dict: A dictionary containing the calculated statistics.
            - For numerical columns: mean, median, std, skewness.
            - For categorical columns: mode.

    Raises:
        TypeError: If df is not a pandas DataFrame.
        KeyError: If column_name does not exist in df.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Company-grade Error: 'df' must be a pandas DataFrame.")

    if column_name not in df.columns:
        raise KeyError(f"Company-grade Error: Column '{column_name}' not found in DataFrame.")

    column = df[column_name]

    # Check if the column is numerical
    if pd.api.types.is_numeric_dtype(column):
        mean_val = column.mean()
        median_val = column.median()
        std_val = column.std()

        # Determine skewness using a safe float comparison
        if np.isclose(mean_val, median_val):
            skewness = "Approximately Symmetric"
        elif mean_val > median_val:
            skewness = "Right Skewed"
        else:
            skewness = "Left Skewed"

        return {
            "column": column_name,
            "type": "numerical",
            "mean": mean_val,
            "median": median_val,
            "std": std_val,
            "skewness": skewness
        }

    # Otherwise, treat the column as categorical
    else:
        mode_val = column.mode()
        if mode_val.empty:
            raise ValueError(f"Company-grade Error: Column '{column_name}' has no valid values to compute mode.")

        return {
            "column": column_name,
            "type": "categorical",
            "mode": mode_val.iloc[0]
        }


# ---------------- Test Dataset ----------------
data = {
    'Transaction_ID': range(1, 11),
    'Product_Category': [
        'Electronics', 'Home', 'Electronics', 'Sports', 'Home',
        'Electronics', 'Home', 'Sports', 'Electronics', 'Electronics'
    ],
    'Sales_Amount': [
        150, 200, 155, 300, 210,
        180, 205, 1000, 190, 160
    ],
    'Customer_Age': [
        25, 34, np.nan, 45, 23,
        31, 29, np.nan, 38, 40
    ],
    'Rating': [
        5, 4, 3, 5, 2,
        4, 5, 2, 4, 3
    ]
}

df_test = pd.DataFrame(data)
df_test.to_csv("company_sales_test.csv", index=False)

# ---------------- Tests ----------------
print(automated_stat_analyzer(df_test, "Sales_Amount"))
print(automated_stat_analyzer(df_test, "Product_Category"))

{'column': 'Sales_Amount', 'type': 'numerical', 'mean': np.float64(275.0), 'median': np.float64(195.0), 'std': np.float64(258.30645021412494), 'skewness': 'Right Skewed'}
{'column': 'Product_Category', 'type': 'categorical', 'mode': 'Electronics'}


### Part 2 - Null handling strategy

In [2]:
def null_handling_strategy(df, strategy="fill_mean"):
    """
    Handle missing values in a DataFrame using the selected strategy.

    Args:
        df (pd.DataFrame): The input DataFrame.
        strategy (str): The missing-value strategy to apply.
            One of "drop_rows", "fill_mean", "fill_median".

    Returns:
        pd.DataFrame: A cleaned copy of the DataFrame.

    Raises:
        TypeError: If df is not a pandas DataFrame.
        ValueError: If the strategy is not supported.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Company-grade Error: 'df' must be a pandas DataFrame.")

    valid_strategies = ["drop_rows", "fill_mean", "fill_median"]
    if strategy not in valid_strategies:
        raise ValueError(
            f"Company-grade Error: Invalid strategy '{strategy}'. "
            f"Supported strategies are: 'drop_rows', 'fill_mean', 'fill_median'."
        )

    # Work on a copy so the original DataFrame is not modified
    result = df.copy()

    # Check for null values first
    null_counts = result.isnull().sum()

    if strategy == "drop_rows":
        result = result.dropna()

    else:
        # Select only the numerical columns
        numeric_columns = result.select_dtypes(include=np.number).columns

        for column in numeric_columns:
            if strategy == "fill_mean":
                result[column] = result[column].fillna(result[column].mean())
            elif strategy == "fill_median":
                result[column] = result[column].fillna(result[column].median())

    return result


# ---------------- Tests ----------------
result1 = null_handling_strategy(df_test, "drop_rows")
result2 = null_handling_strategy(df_test, "fill_mean")
result3 = null_handling_strategy(df_test, "fill_median")

print("drop_rows:")
print(result1)
print()
print("fill_mean:")
print(result2)
print()
print("fill_median:")
print(result3)

drop_rows:
   Transaction_ID Product_Category  Sales_Amount  Customer_Age  Rating
0               1      Electronics           150          25.0       5
1               2             Home           200          34.0       4
3               4           Sports           300          45.0       5
4               5             Home           210          23.0       2
5               6      Electronics           180          31.0       4
6               7             Home           205          29.0       5
8               9      Electronics           190          38.0       4
9              10      Electronics           160          40.0       3

fill_mean:
   Transaction_ID Product_Category  Sales_Amount  Customer_Age  Rating
0               1      Electronics           150        25.000       5
1               2             Home           200        34.000       4
2               3      Electronics           155        33.125       3
3               4           Sports           300      